# TEMA-only top-10 book (IS rank → OOS gate)

KovaView production TEMA stack: **9/99/199** regime/trigger + **MACD(12,26,9) close**, **2.5× ATR** stop (see `/cash` and `docs/SWING_STRATEGY.md`).

This notebook:
1. Scans the liquid US universe (same names as `LIQUID_NAMES` in the web app).
2. Ranks tickers by **in-sample** (60%) TEMA Sharpe — no OOS peeking for selection.
3. Builds an equal-weight **top-10** book on the **out-of-sample** (40%) window.
4. **Promotes** the list for a TEMA-only FTMO/manual book **only if** the OOS gate passes.

Research only — not live orders.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from pipeline.research.tema_top10_oos import (
    oos_positive_alternates,
    portfolio_equity_curve,
    run_top10,
)
from pipeline.research.kama_backtest import download_close
from pipeline.compute.perps_math import TEMA_SL_ATR, tema_notional, wilder_atr

START = "2018-01-01"
TOP_N = 10
SELECTION = "is_rank"  # or "fixed" for megacap FIXED_TOP10 list
ACCOUNT_EQUITY = 100_000.0  # rescale for your FTMO challenge size


In [ ]:
result = run_top10(start=START, top_n=TOP_N, selection=SELECTION)
print(result.gate_reason)
print("Gate checks:", result.gate_checks)
print("Top-10 (IS rank):", result.top10)
if result.portfolio_oos:
    print("Portfolio OOS Sharpe:", round(result.portfolio_oos["sharpe"], 2))
    print("Equal-weight B&H OOS Sharpe:", round(result.portfolio_bh_oos["sharpe"], 2))

PROMOTED = result.top10 if result.gate_passed else []
print("\nPROMOTED for TEMA-only trading:", PROMOTED or "(none — gate failed)")


In [ ]:
rank_df = pd.DataFrame([{
    "ticker": r.ticker,
    "is_sharpe": r.is_sharpe,
    "oos_sharpe": r.oos_sharpe,
    "oos_cagr": r.oos_cagr,
    "oos_max_dd": r.oos_max_dd,
    "in_top10": r.ticker in result.top10,
} for r in result.ranked])
rank_df.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_df = rank_df.head(15)
x = range(len(plot_df))
ax.bar([i - 0.2 for i in x], plot_df["is_sharpe"], width=0.4, label="IS Sharpe")
ax.bar([i + 0.2 for i in x], plot_df["oos_sharpe"], width=0.4, label="OOS Sharpe")
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df["ticker"], rotation=45, ha="right")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title(f"TEMA Sharpe by ticker (top 15 by IS rank) — gate {'PASS' if result.gate_passed else 'FAIL'}")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
if result.top10:
    closes = {t: download_close(t, START) for t in result.top10}
    curve = portfolio_equity_curve(result.top10, closes)
    if not curve.empty:
        ax = curve.plot(figsize=(10, 4), title="Equal-weight OOS equity (top-10 selection)")
        ax.set_ylabel("Growth of $1")
        plt.show()


In [ ]:
if not result.gate_passed:
    alts = oos_positive_alternates(result)
    print("Gate failed — do not trade the IS-ranked top-10 blindly.")
    print("OOS-positive names in universe (monitoring only, not promoted):")
    for r in alts:
        print(f"  {r.ticker}: OOS Sharpe {r.oos_sharpe:.2f}")
    print("\nProduction app uses 3 long / 3 short TEMA slots on /cash — smaller book than top-10.")
else:
    slot = ACCOUNT_EQUITY / len(PROMOTED)
    rows = []
    for t in PROMOTED:
        c = download_close(t, START)
        close = c.to_numpy(dtype=float)
        h = close * 1.004
        l = close * 0.996
        atr = wilder_atr(h, l, close)
        last = float(close[-1])
        notional = tema_notional(slot, last, atr)
        shares = notional / last if last else 0
        stop = last - TEMA_SL_ATR * atr
        rows.append({"ticker": t, "close": last, "shares": shares, "stop": stop, "slot_usd": slot})
    sizing = pd.DataFrame(rows)
    sizing


## OOS gate thresholds

Edit `OOS_GATE` in `pipeline/research/tema_top10_oos.py` if you want stricter promotion rules.

Current defaults: `{'min_portfolio_oos_sharpe': 0.35, 'min_median_ticker_oos_sharpe': 0.0, 'min_fraction_oos_sharpe_positive': 0.55, 'min_portfolio_oos_cagr_vs_bh': -0.05}`

Re-run CLI: `PYTHONPATH=. python -m pipeline.research.tema_top10_oos --selection fixed` to test the fixed megacap list `('AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'JPM', 'XOM', 'JNJ', 'V')`.